In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/monthly_vehicle_demand.csv"
)

df["month"] = pd.to_datetime(df["month"])

print(df.shape)
df.head()

(23650, 5)


,month,state,vehicle_type,registrations,vehicle_group
0,2019-01-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,9.0,Goods Vehicle
1,2019-01-01,Andaman And Nicobar Islands,Heavy Passenger Vehicle,1.0,Passenger Transport
2,2019-01-01,Andaman And Nicobar Islands,Light Goods Vehicle,21.0,Goods Vehicle
3,2019-01-01,Andaman And Nicobar Islands,Light Motor Vehicle,161.0,Passenger Vehicle
4,2019-01-01,Andaman And Nicobar Islands,Light Passenger Vehicle,27.0,Passenger Vehicle


In [2]:
df = df.sort_values(
    ["state", "vehicle_group", "month"]
).reset_index(drop=True)

In [3]:
df["target_demand"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .shift(-1)
)

In [4]:
df["lag_1"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .shift(1)
)

In [5]:
df["lag_2"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .shift(2)
)

In [6]:
df["lag_3"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .shift(3)
)

In [7]:
df["lag_12"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .shift(12)
)

In [8]:
df["rolling_mean_3"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [9]:
df["rolling_mean_6"] = (
    df.groupby(
        ["state", "vehicle_group"]
    )["registrations"]
    .transform(
        lambda x: x.shift(1).rolling(6).mean()
    )
)

In [10]:
df["monthly_growth"] = (
    (df["registrations"] - df["lag_1"])
    / df["lag_1"].replace(0, np.nan)
)

In [11]:
df["yearly_growth"] = (
    (df["registrations"] - df["lag_12"])
    / df["lag_12"].replace(0, np.nan)
)

In [12]:
df["year"] = df["month"].dt.year

df["month_number"] = df["month"].dt.month

df["quarter"] = df["month"].dt.quarter

In [13]:
df.head(20)

,month,state,vehicle_type,registrations,vehicle_group,target_demand,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,monthly_growth,yearly_growth,year,month_number,quarter
0,2019-01-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,9.0,Goods Vehicle,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019,1,1
1,2019-01-01,Andaman And Nicobar Islands,Light Goods Vehicle,21.0,Goods Vehicle,1.0,9.0,NaN,NaN,NaN,NaN,NaN,1.333333,NaN,2019,1,1
2,2019-01-01,Andaman And Nicobar Islands,Medium Goods Vehicle,1.0,Goods Vehicle,6.0,21.0,9.0,NaN,NaN,NaN,NaN,-0.952381,NaN,2019,1,1
3,2019-02-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,6.0,Goods Vehicle,18.0,1.0,21.0,9.0,NaN,10.333333,NaN,5.000000,NaN,2019,2,1
4,2019-02-01,Andaman And Nicobar Islands,Light Goods Vehicle,18.0,Goods Vehicle,1.0,6.0,1.0,21.0,NaN,9.333333,NaN,2.000000,NaN,2019,2,1
5,2019-02-01,Andaman And Nicobar Islands,Medium Goods Vehicle,1.0,Goods Vehicle,1.0,18.0,6.0,1.0,NaN,8.333333,NaN,-0.944444,NaN,2019,2,1
6,2019-03-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,1.0,Goods Vehicle,5.0,1.0,18.0,6.0,NaN,8.333333,9.333333,0.000000,NaN,2019,3,1
7,2019-03-01,Andaman And Nicobar Islands,Light Goods Vehicle,5.0,Goods Vehicle,22.0,1.0,1.0,18.0,NaN,6.666667,8.000000,4.000000,NaN,2019,3,1
8,2019-04-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,22.0,Goods Vehicle,6.0,5.0,1.0,1.0,NaN,2.333333,5.333333,3.400000,NaN,2019,4,2
9,2019-04-01,Andaman And Nicobar Islands,Light Goods Vehicle,6.0,Goods Vehicle,1.0,22.0,5.0,1.0,NaN,9.333333,8.833333,-0.727273,NaN,2019,4,2


In [14]:
print(df.columns)

Index(['month', 'state', 'vehicle_type', 'registrations', 'vehicle_group',
       'target_demand', 'lag_1', 'lag_2', 'lag_3', 'lag_12', 'rolling_mean_3',
       'rolling_mean_6', 'monthly_growth', 'yearly_growth', 'year',
       'month_number', 'quarter'],
      dtype='object')


In [15]:
df_model = df.dropna(
    subset=[
        "target_demand",
        "lag_1",
        "lag_2",
        "lag_3",
        "lag_12",
        "rolling_mean_3",
        "rolling_mean_6"
    ]
).copy()

In [16]:
print("Original rows:", len(df))
print("ML rows:", len(df_model))

Original rows: 23650
ML rows: 21021


In [17]:
df_model.to_csv(
    "../data/processed/ml_vehicle_demand.csv",
    index=False
)

print("ML dataset saved successfully!")

ML dataset saved successfully!


In [18]:
print(df_model.shape)

(21021, 17)


In [19]:
df_model.head(10)

,month,state,vehicle_type,registrations,vehicle_group,target_demand,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,monthly_growth,yearly_growth,year,month_number,quarter
12,2019-05-01,Andaman And Nicobar Islands,Light Goods Vehicle,1.0,Goods Vehicle,1.0,5.0,1.0,6.0,9.0,4.000000,6.666667,-0.800000,-0.888889,2019,5,2
13,2019-05-01,Andaman And Nicobar Islands,Medium Goods Vehicle,1.0,Goods Vehicle,1.0,1.0,5.0,1.0,21.0,2.333333,6.666667,0.000000,-0.952381,2019,5,2
14,2019-06-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,1.0,Goods Vehicle,3.0,1.0,1.0,5.0,1.0,2.333333,6.000000,0.000000,0.000000,2019,6,2
15,2019-06-01,Andaman And Nicobar Islands,Light Goods Vehicle,3.0,Goods Vehicle,2.0,1.0,1.0,1.0,6.0,1.000000,2.500000,2.000000,-0.500000,2019,6,2
16,2019-07-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,2.0,Goods Vehicle,5.0,3.0,1.0,1.0,18.0,1.666667,2.000000,-0.333333,-0.888889,2019,7,3
17,2019-07-01,Andaman And Nicobar Islands,Light Goods Vehicle,5.0,Goods Vehicle,1.0,2.0,3.0,1.0,1.0,2.000000,2.166667,1.500000,4.000000,2019,7,3
18,2019-07-01,Andaman And Nicobar Islands,Medium Goods Vehicle,1.0,Goods Vehicle,4.0,5.0,2.0,3.0,1.0,3.333333,2.166667,-0.800000,0.000000,2019,7,3
19,2019-08-01,Andaman And Nicobar Islands,Heavy Goods Vehicle,4.0,Goods Vehicle,3.0,1.0,5.0,2.0,5.0,2.666667,2.166667,3.000000,-0.200000,2019,8,3
20,2019-08-01,Andaman And Nicobar Islands,Light Goods Vehicle,3.0,Goods Vehicle,1.0,4.0,1.0,5.0,22.0,3.333333,2.666667,-0.250000,-0.863636,2019,8,3
21,2019-08-01,Andaman And Nicobar Islands,Medium Goods Vehicle,1.0,Goods Vehicle,1.0,3.0,4.0,1.0,6.0,2.666667,3.000000,-0.666667,-0.833333,2019,8,3
